In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 5.5
fig_height = 3.5
fig_format = 'pdf'
fig_dpi = 300
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL2VzbWVyYWxkYWxhcnNzb24vRG9jdW1lbnRzL0dpdEh1Yi9BRUYtZ2l0L0Vrc2FtZW4='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/Users/esmeraldalarsson/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/importlib/_bootstrap.py": 1772783774.8312185, "/Users/esmeraldalarsson/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/importlib/_bootstrap_external.py": 1772783774.8374882, "/Users/esmeraldalarsson/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/zipimport.py": 1772783777.224686, "/Users/esmeraldalarsson/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/codecs.py": 1772783772.361852, "/Users/esmeraldalarsson/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/encodings/aliases.py": 1772783772.403677, "/Users/esmeraldalarsson/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/encodings/__init__.py": 1772783772.4024942, "/Users/esmeraldalarsson/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/encodings/utf_8.py": 1772783772.4356892, "/Users/esmeraldalar

In [2]:
# standard library
from pathlib import Path
import itertools

# data handling and model storage
import joblib
import numpy as np
import pandas as pd
import tidyfinance as tf

# plotting
from plotnine import *
from mizani.formatters import percent_format
from IPython.display import Markdown
import matplotlib.pyplot as plt

# statistical models
import statsmodels.api as sm

# machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.covariance import LedoitWolf

In [3]:
# ------------------------------------------------------------
# Load high-frequency minute price data
# ------------------------------------------------------------

data_path = Path("/Users/esmeraldalarsson/Desktop/hf_minute_prices.parquet")

hf_prices = pd.read_parquet(data_path)

# Convert timestamp column to datetime format
hf_prices["ts"] = pd.to_datetime(hf_prices["ts"])

# Extract trading day from timestamp
hf_prices["trading_day"] = hf_prices["ts"].dt.date


# ------------------------------------------------------------
# Fixed trading-hours setup used
# ------------------------------------------------------------

MARKET_OPEN = "09:30:00"
MARKET_CLOSE = "15:59:00"

# Preserve the timezone of the timestamp column.
# In the rendered output, timestamps appear as +00:00,
# so the grid must use the same timezone.
TS_TZ = hf_prices["ts"].dt.tz

def market_timestamp(trading_day, clock_time):
    """
    Create a timestamp for a given trading day and clock time,
    using the same timezone as the raw timestamp column.
    """
    return pd.Timestamp(f"{trading_day} {clock_time}", tz=TS_TZ)

In [4]:
# ------------------------------------------------------------
# Keep AAPL only and sort by timestamp
# ------------------------------------------------------------

aapl_raw = (
    hf_prices
    .query("symbol == 'AAPL'")
    .sort_values(["trading_day", "ts"])
    .copy()
)

# Compute log price
# High-frequency realized variance is based on log returns
aapl_raw["log_price"] = np.log(aapl_raw["price"])

In [5]:
# ------------------------------------------------------------
# Check missing minute observations before forward-filling
# ------------------------------------------------------------

def missing_minutes_one_day(group):
    """
    Count missing minute observations within one trading day.

    Missing observations are measured relative to a fixed regular
    trading-hours grid from 09:30 to 15:59.

    The grid uses the same timezone as the timestamp column.
    """
    
    group = group.sort_values("ts").copy()
    trading_day = group["trading_day"].iloc[0]

    market_open = market_timestamp(trading_day, MARKET_OPEN)
    market_close = market_timestamp(trading_day, MARKET_CLOSE)

    full_grid = pd.date_range(
        start=market_open,
        end=market_close,
        freq="1min"
    )

    observed_minutes = group["ts"].drop_duplicates()

    expected_minutes = len(full_grid)
    observed_minutes_count = observed_minutes.nunique()
    missing_minutes = expected_minutes - observed_minutes_count
    missing_share = missing_minutes / expected_minutes

    return pd.Series({
        "market_open": market_open,
        "market_close": market_close,
        "first_observed_timestamp": group["ts"].min(),
        "last_observed_timestamp": group["ts"].max(),
        "expected_minutes": expected_minutes,
        "observed_minutes": observed_minutes_count,
        "missing_minutes": missing_minutes,
        "missing_share": missing_share
    })


aapl_missing_by_day = (
    aapl_raw
    .groupby("trading_day")
    .apply(missing_minutes_one_day)
    .reset_index()
)

overall_missing_share = (
    aapl_missing_by_day["missing_minutes"].sum()
    / aapl_missing_by_day["expected_minutes"].sum()
)

/var/folders/kn/wj00sq0n2szb0bsk4n7_q4tr0000gn/T/ipykernel_42362/3087448129.py:49: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


In [6]:
# ------------------------------------------------------------
# Create regular one-minute grid and forward-fill missing prices
# ------------------------------------------------------------

# Create the fixed intraday minute grid.
intraday_times = pd.date_range(
    start=f"2000-01-01 {MARKET_OPEN}",
    end=f"2000-01-01 {MARKET_CLOSE}",
    freq="1min"
).time

# Use all AAPL trading days
trading_days = np.sort(aapl_raw["trading_day"].unique())

# Create a full grid of every trading day x every intraday minute
grid = pd.MultiIndex.from_product(
    [trading_days, intraday_times],
    names=["trading_day", "time"]
).to_frame(index=False)

# Create timezone-aware timestamps for the full grid
grid["ts"] = pd.to_datetime(
    grid["trading_day"].astype(str) + " " + grid["time"].astype(str)
)

if TS_TZ is not None:
    grid["ts"] = grid["ts"].dt.tz_localize(TS_TZ)

# Prepare observed AAPL prices for merging onto the grid
aapl_for_merge = (
    aapl_raw
    .copy()
    .assign(time=lambda x: x["ts"].dt.time)
    [["trading_day", "time", "price"]]
)

# Merge observed prices onto the fixed grid
aapl_minute = (
    grid
    .merge(
        aapl_for_merge,
        on=["trading_day", "time"],
        how="left"
    )
    .sort_values(["trading_day", "ts"])
    .copy()
)

aapl_minute["symbol"] = "AAPL"

# Previous-tick interpolation within each trading day only
aapl_minute["price"] = (
    aapl_minute
    .groupby("trading_day")["price"]
    .ffill()
)

# Drop missing prices before the first observed price of a day.
# We do not backfill.
aapl_minute = (
    aapl_minute
    .dropna(subset=["price"])
    .sort_values(["trading_day", "ts"])
    .reset_index(drop=True)
)

aapl_minute["log_price"] = np.log(aapl_minute["price"])

minute_check = (
    aapl_minute
    .groupby("trading_day")
    .size()
    .rename("n_minutes_after_fill")
    .reset_index()
)

In [7]:
# ------------------------------------------------------------
# Function to compute daily realized variance
# ------------------------------------------------------------

def compute_daily_rv(data, sampling_minutes):
    """
    Compute daily realized variance for a given sampling interval.

    Prices are sampled on a fixed grid relative to the market open.
    For example, 10-minute sampling keeps 09:30, 09:40, 09:50, ...
    """
    
    sampled = data.sort_values(["trading_day", "ts"]).copy()

    sampled["market_open"] = sampled["trading_day"].apply(
        lambda d: market_timestamp(d, MARKET_OPEN)
    )

    sampled["minutes_since_open"] = (
        (sampled["ts"] - sampled["market_open"])
        .dt.total_seconds()
        / 60
    ).round().astype(int)

    # Keep only observations on the Delta-minute grid
    sampled = sampled[
        sampled["minutes_since_open"] % sampling_minutes == 0
    ].copy()

    # Compute intraday log returns within each trading day
    sampled["intraday_return"] = (
        sampled
        .groupby("trading_day")["log_price"]
        .diff()
    )

    rv_daily = (
        sampled
        .groupby("trading_day")
        .agg(
            realized_variance=(
                "intraday_return",
                lambda x: np.nansum(x ** 2)
            ),
            n_returns=(
                "intraday_return",
                lambda x: x.notna().sum()
            )
        )
        .reset_index()
    )

    rv_daily["sampling_minutes"] = sampling_minutes

    return rv_daily

In [8]:
# ------------------------------------------------------------
# Compute realized variance for all sampling intervals
# ------------------------------------------------------------

sampling_intervals = [1, 2, 5, 10, 15, 30, 60]

rv_all = pd.concat(
    [
        compute_daily_rv(aapl_minute, interval)
        for interval in sampling_intervals
    ],
    ignore_index=True
)

In [9]:
# ------------------------------------------------------------
# Average realized variance and estimation uncertainty
# ------------------------------------------------------------
rv_summary = (
    rv_all
    .groupby("sampling_minutes")
    .agg(
        mean_rv=("realized_variance", "mean"),
        sd_rv=("realized_variance", "std"),
        n_days=("realized_variance", "count")
    )
    .reset_index()
)

rv_summary["se_rv"] = rv_summary["sd_rv"] / np.sqrt(rv_summary["n_days"])

rv_summary["ci_lower"] = rv_summary["mean_rv"] - 1.96 * rv_summary["se_rv"]
rv_summary["ci_upper"] = rv_summary["mean_rv"] + 1.96 * rv_summary["se_rv"]

In [10]:
#| label: fig-aapl-volatility-signature
#| fig-cap: Volatility signature plot for AAPL with 95% confidence intervals.
#| fig-align: center

# ------------------------------------------------------------
# Volatility signature plot with error bars
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.errorbar(
    rv_summary["sampling_minutes"],
    rv_summary["mean_rv"],
    yerr=1.96 * rv_summary["se_rv"],
    marker="o",
    capsize=4,
    linewidth=1.5,
    label="Average daily realized variance"
)

plt.xlabel("Sampling interval $\\Delta$ in minutes")
plt.ylabel("Average daily realized variance")
plt.title("Volatility signature plot for AAPL")
plt.xticks(sampling_intervals)
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

<Figure size 2400x1500 with 1 Axes>

In [11]:
# ------------------------------------------------------------
# Exercise 2a: Daily realized covariance matrices
# ------------------------------------------------------------

# Use the sampling interval chosen in Exercise 1
DELTA = 5

symbols = np.sort(hf_prices["symbol"].unique())
trading_days_all = np.sort(hf_prices["trading_day"].unique())

# Create one-minute intraday grid
intraday_times = pd.date_range(
    start=f"2000-01-01 {MARKET_OPEN}",
    end=f"2000-01-01 {MARKET_CLOSE}",
    freq="1min"
).time

# Full grid: trading day x intraday minute
grid = pd.MultiIndex.from_product(
    [trading_days_all, intraday_times],
    names=["trading_day", "time"]
).to_frame(index=False)

grid["ts"] = pd.to_datetime(
    grid["trading_day"].astype(str) + " " + grid["time"].astype(str)
)

if TS_TZ is not None:
    grid["ts"] = grid["ts"].dt.tz_localize(TS_TZ)

grid["minutes_since_open"] = (
    grid
    .groupby("trading_day")
    .cumcount()
)

full_index = pd.MultiIndex.from_frame(
    grid[["trading_day", "ts"]]
)

# Put prices in wide format once:
# rows = trading day and timestamp
# columns = stocks
prices_wide_raw = (
    hf_prices
    .pivot_table(
        index=["trading_day", "ts"],
        columns="symbol",
        values="price",
        aggfunc="last"
    )
    .sort_index()
)

# Reindex to full one-minute grid
prices_wide = (
    prices_wide_raw
    .reindex(full_index)
    .sort_index()
)

# Previous-tick interpolation within each trading day only
prices_wide = (
    prices_wide
    .groupby(level="trading_day")
    .ffill()
)

# Fixed stock order
prices_wide = prices_wide[symbols]

# Select 5-minute timestamps
sample_grid = grid.loc[
    grid["minutes_since_open"] % DELTA == 0,
    ["trading_day", "ts"]
]

sample_index = pd.MultiIndex.from_frame(sample_grid)

# Sample log prices
sampled_log_prices = np.log(
    prices_wide.loc[sample_index]
)

# Intraday log returns within each trading day
returns_wide = (
    sampled_log_prices
    .groupby(level="trading_day")
    .diff()
)

# Compute daily realized covariance matrices
rc_matrices = {}
rc_diagnostics = []

for trading_day, returns_day in returns_wide.groupby(level="trading_day"):
    
    returns_day = returns_day.droplevel("trading_day")
    returns_day = returns_day.dropna(how="any")
    
    if len(returns_day) < 2:
        continue
    
    R = returns_day[symbols].to_numpy()
    
    # Realized covariance matrix:
    # RC_d = sum_j r_{d,j} r_{d,j}'
    rc_day = R.T @ R
    
    rc_matrices[trading_day] = pd.DataFrame(
        rc_day,
        index=symbols,
        columns=symbols
    )
    
    rc_diagnostics.append({
        "trading_day": trading_day,
        "n_returns": len(returns_day),
        "matrix_rank": np.linalg.matrix_rank(rc_day),
        "min_eigenvalue": np.linalg.eigvalsh(rc_day).min()
    })

rc_diagnostics = pd.DataFrame(rc_diagnostics)

In [12]:
# ------------------------------------------------------------
# Exercise 2b: Extract diagonal elements from RC_d
# ------------------------------------------------------------

rv_by_stock = pd.DataFrame(
    [
        {
            "trading_day": day,
            **dict(zip(symbols, np.diag(rc_matrix.values)))
        }
        for day, rc_matrix in rc_matrices.items()
    ]
)

rv_by_stock["trading_day"] = pd.to_datetime(rv_by_stock["trading_day"])

rv_long = (
    rv_by_stock
    .melt(
        id_vars="trading_day",
        value_vars=symbols,
        var_name="symbol",
        value_name="realized_variance"
    )
)

In [13]:
#| label: fig-realized-variances-30-stocks
#| fig-cap: Daily realized variances for the 30 DJIA stocks.
#| fig-align: center
#| fig-pos: H

import matplotlib.pyplot as plt
import numpy as np

plt.close("all")

rv_long_plot = rv_long.sort_values(["symbol", "trading_day"]).copy()

fig, ax = plt.subplots(figsize=(14, 7))

# 30 forskellige farver
colors = plt.cm.nipy_spectral(
    np.linspace(0, 1, len(symbols))
)

for color, symbol in zip(colors, symbols):

    stock_data = rv_long_plot[
        rv_long_plot["symbol"] == symbol
    ]

    ax.plot(
        stock_data["trading_day"],
        stock_data["realized_variance"],
        label=symbol,
        color=color,
        linewidth=0.8,
        alpha=0.7
    )

ax.set_xlabel("Trading day")
ax.set_ylabel("Realized variance")
ax.set_title("Daily realized variances for the 30 DJIA stocks")

ax.grid(True, alpha=0.3)

# Legend i øverste højre hjørne
ax.legend(
    loc="upper right",
    ncol=3,
    fontsize=7,
    frameon=True,
    framealpha=0.9
)

plt.xticks(rotation=45)

fig.tight_layout()

plt.show()

<Figure size 4200x2100 with 1 Axes>